Notebook to calculate total cases, peak height, peak week, and $R_0$ for each model's predictions.

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as st
import mosqlient as mosq
from epiweeks import Week
from aux_func import get_data
from analysis import * 
import matplotlib.pyplot as plt 
import seaborn as sns 
from mosqlient.prediction_optimize import get_df_pars_ls

In [3]:
code_to_state = {33: 'RJ', 32: 'ES', 41: 'PR', 23: 'CE', 21: 'MA',
 31: 'MG', 42: 'SC', 26: 'PE', 25: 'PB', 24: 'RN', 22: 'PI', 27: 'AL',
 28: 'SE', 35: 'SP', 43: 'RS', 15: 'PA', 16: 'AP', 14: 'RR',  11: 'RO',
 13: 'AM', 12: 'AC', 51: 'MT', 50: 'MS', 52: 'GO', 17: 'TO', 53: 'DF',
 29: 'BA'}

state_to_code = {value: key for key, value in code_to_state.items()}

geo_dengue = [2931350,2933307,2302503,3119401,
              3549805,3541406,1200401,1200203,
              1716109,4113700,4103701,4104808,
              5201405,5102637,5215231]

geo_chik = [2211001,2931350,3143302,3119401,
            1721000,1716109,4104808,4219507,
            5103403,5102637] 

In [ ]:
df_dengue_state  = get_data('dengue_state')
df_dengue_state.date = pd.to_datetime(df_dengue_state.date)

In [5]:
challenge = 'dengue_state'

df_preds = pd.read_csv(f'./predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
1,2022-10-16,37.019327,43.295517,51.743385,66.653189,83.207109,100.790847,116.992117,126.864602,134.406880,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
2,2022-10-23,55.945220,68.411390,84.200302,108.887826,135.698419,163.344639,189.017119,205.765306,218.575663,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
3,2022-10-30,58.185781,69.697014,83.792687,107.076328,132.601767,158.745145,184.276095,201.204977,212.912726,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
4,2022-11-06,58.634062,68.591352,80.855678,101.969117,125.548661,150.259612,174.517426,190.524410,202.512661,12,6822,1,95.67,3rd_imdc_isi_isi-dengue


In [7]:
df_dengue_state = df_dengue_state.merge(
        df_preds[["date", "adm_1", "validation"]].drop_duplicates(
                subset=["date", "adm_1", "validation"],
                keep="first"
            ), on=["date", "adm_1"]
    )

df_dengue_state.head()

,date,adm_1,casos,validation
0,2022-10-09,11,99,1
1,2022-10-09,12,31,1
2,2022-10-09,13,250,1
3,2022-10-09,14,2,1
4,2022-10-09,15,60,1


In [8]:
def process_single_combination(df_preds, df_dengue_state, state, model, validation, n_paths=1000, n_samples=100, seed=0):
    """
    Processa a simulação e métricas para uma única combinação de modelo e validação.
    """
    rng = np.random.default_rng(seed)
    
    # 1. Filtragem e preparação de parâmetros
    df_preds_m = df_preds.loc[(df_preds.model == model) & (df_preds.adm_1 == state)]
    if df_preds_m.empty:
        return None  # Retorna None se não houver dados para essa combinação
        
    df_pars = get_df_pars_ls(df_preds_m)
    df_pars.loc[df_pars.sigma == 0, 'sigma'] = 0.1

    # 2. Estimativa de Rho e Marginais
    df_rho_input = df_pars.loc[(df_pars.validation == validation - 1)].merge(
        df_dengue_state, on=['date', 'adm_1', 'validation'], how='left'
    )
    rho = estimate_rho(df_rho_input)
    marginals_uf = build_marginals(df_pars, state, validation)

    # 3. Geração das trajetórias (paths)
    paths = np.vstack([
        sample_path(marginals_uf, rho, random_state=rng)
        for _ in range(n_paths)
    ])

    # 4. Cálculo das métricas baseadas em todas as trajetórias
    season_total = paths.sum(axis=1)
    season_peak = paths.max(axis=1)

    # 5. Amostragem para ajuste da curva de Richards
    indices_aleatorios = rng.choice(paths.shape[0], size=n_samples, replace=False)
    amostras_paths = paths[indices_aleatorios, :]

    r0_dist = []
    pico_dist = []
    for path in amostras_paths:
        r0_i, pico_i = otim_single_path(path)
        r0_dist.append(r0_i)
        pico_dist.append(pico_i)

    r0_dist = np.array(r0_dist)
    pico_dist = np.array(pico_dist)

    # 6. Consolidação dos resultados em um dicionário de intervalos
    metrics = {
        'model': model,
        'validation': validation,
        'state': state
    }
    
    # Dicionário mapeando o nome da métrica para o seu array de distribuição
    distributions = {
        'season_total': season_total,
        'season_peak': season_peak,
        'r0_dist': r0_dist,
        'pico_dist': pico_dist
    }
    
    # Calcula os quantis automaticamente para cada métrica
    for name, dist in distributions.items():
        lo, med, hi = np.quantile(dist, [0.025, 0.5, 0.975])
        metrics[f'{name}_p25'] = lo
        metrics[f'{name}_p50'] = med
        metrics[f'{name}_p975'] = hi

    return metrics

In [15]:
def run_pipeline(df_preds, df_dengue_state, state, models_list, validations_list):
    """
    Executa o loop por todos os modelos e validações e retorna um DataFrame consolidado.
    """
    results = []
    
    for model in models_list:
        for validation in validations_list:
            #print(f"Processando: Modelo={model} | Validation={validation}...")
            try:
                res = process_single_combination(df_preds, df_dengue_state, state, model, validation)
                if res is not None:
                    results.append(res)
            except Exception as e:
                print(f"Erro ao processar Modelo={model}, Validation={validation}: {e}, State: {state}")
                continue
                
    # Transforma a lista de dicionários em um DataFrame estruturado
    df_results = pd.DataFrame(results)
    return df_results

In [9]:
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
1,2022-10-16,37.019327,43.295517,51.743385,66.653189,83.207109,100.790847,116.992117,126.864602,134.406880,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
2,2022-10-23,55.945220,68.411390,84.200302,108.887826,135.698419,163.344639,189.017119,205.765306,218.575663,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
3,2022-10-30,58.185781,69.697014,83.792687,107.076328,132.601767,158.745145,184.276095,201.204977,212.912726,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
4,2022-11-06,58.634062,68.591352,80.855678,101.969117,125.548661,150.259612,174.517426,190.524410,202.512661,12,6822,1,95.67,3rd_imdc_isi_isi-dengue


In [11]:
df_params_old = pd.read_csv(f'predictions/new_metrics_{challenge}.csv.gz', index_col = 'Unnamed: 0')

df_params_old.head()

,model,validation,state,season_total_p25,season_total_p50,season_total_p975,season_peak_p25,season_peak_p50,season_peak_p975,r0_dist_p25,r0_dist_p50,r0_dist_p975,pico_dist_p25,pico_dist_p50,pico_dist_p975
0,3rd_imdc_isi_isi-dengue,2,12,8155.220843,9666.404427,11588.776658,225.289613,295.884608,419.352327,1.255347,1.277699,1.304268,29.934870,31.934181,35.000000
1,3rd_imdc_isi_isi-dengue,3,12,5296.213880,6433.353223,7836.201734,148.828929,190.215776,248.274815,1.259569,1.275572,1.288510,29.806261,32.044642,34.895799
2,3rd_imdc_isi_isi-dengue,4,12,7235.298114,8421.714410,9572.018224,199.337666,244.680701,301.015333,1.259767,1.273151,1.289039,30.060873,31.744612,34.252873
3,3rd_imdc_purdue_neuralearth,2,12,2120.596418,6765.841749,28221.394693,144.663450,611.092154,4306.154039,1.109418,1.536371,2.135155,7.888542,18.482081,33.183729
4,3rd_imdc_purdue_neuralearth,3,12,1518.752445,5766.735068,25714.256030,98.427024,478.604529,3005.435595,1.236723,1.570767,2.050592,11.531725,19.721003,31.194712


In [13]:
lista_modelos = np.setdiff1d(df_preds.model.unique(), df_params_old.model)

lista_modelos

array(['3rd_imdc_afya_ric', '3rd_imdc_lncc_clidengo26dengue',
       '3rd_imdc_rki_rki_zki_ph', '3rd_imdc_rki_rki_zki_ph_lstm_geo',
       '3rd_imdc_unesp_recogna'], dtype=object)

In [16]:
%%time
# Listas de modelos e validações que você quer rodar
#lista_modelos = df_preds.model.unique()
lista_validations = [2,3,4]

states = df_preds.adm_1.unique()


df_all_adm = pd.DataFrame()

for state_alvo in states: 

    # Executa o pipeline
    df_final_resultados = run_pipeline(df_preds, df_dengue_state, state_alvo, lista_modelos, lista_validations)

    # Salva os resultados em um arquivo
    df_all_adm = pd.concat([df_all_adm, df_final_resultados], ignore_index=True) 


df_all_adm.head()

CPU times: user 1h 25min 46s, sys: 957 ms, total: 1h 25min 47s
Wall time: 1h 25min 47s


,model,validation,state,season_total_p25,season_total_p50,season_total_p975,season_peak_p25,season_peak_p50,season_peak_p975,r0_dist_p25,r0_dist_p50,r0_dist_p975,pico_dist_p25,pico_dist_p50,pico_dist_p975
0,3rd_imdc_afya_ric,2,12,2444.906178,4121.603849,10855.326584,223.114065,361.407330,2251.860869,1.191494,2.155109,2.313113,8.578248,9.678033,14.742394
1,3rd_imdc_afya_ric,3,12,6531.048287,10926.542490,24731.664414,372.240768,519.526769,917.983947,1.255681,1.458357,1.527898,16.653453,18.032772,34.388111
2,3rd_imdc_afya_ric,4,12,5638.064605,8784.399308,22266.049303,320.238346,425.263268,664.948352,1.260186,1.564155,1.675024,13.650069,15.324665,29.124032
3,3rd_imdc_lncc_clidengo26dengue,2,12,469.578382,2042.403845,10249.141369,18.579861,89.372373,630.030468,1.211236,1.322364,1.554738,12.233261,27.314862,35.000000
4,3rd_imdc_lncc_clidengo26dengue,3,12,416.245787,1998.991020,9802.374713,13.111074,46.173022,332.529486,1.284650,1.290058,1.346957,20.086080,27.408169,31.688458


In [18]:
df_end = pd.concat([df_params_old, df_all_adm], ignore_index = True)

df_end.to_csv(f'predictions/new_metrics_{challenge}.csv.gz', index = False)

df_all_adm.to_csv(f'predictions/new_metrics_{challenge}.csv.gz')

### Get the True parameters:

In [12]:
df_ = df_dengue_state.merge(df_preds[['date', 'validation']].drop_duplicates(), on = 'date', how = 'left').dropna()

df_.head()

,date,adm_1,casos,validation
17982,2022-10-09,11,99,1.0
17983,2022-10-09,12,31,1.0
17984,2022-10-09,13,250,1.0
17985,2022-10-09,14,2,1.0
17986,2022-10-09,15,60,1.0


In [20]:
from itertools import product
state = 11 
validation = 1 

list_pars = []

for state, validation in product(df_.adm_1.unique(), df_.validation.unique()):  
    df_state = df_.loc[(df_.adm_1 == state) & (df_.validation == validation)].sort_values(by = 'date')

    r0_i, pico_i = otim_single_path(df_state.casos.values)

    total_cases = df_state.casos.sum()
    max_cases = df_state.casos.max()

    list_pars.append(pd.DataFrame([[state, validation, pico_i, total_cases, max_cases]], columns = ['state', 'validation', 
                                                                                                    'peak_week', 'total_cases', 'peak_height']))

df_params = pd.concat(list_pars)

df_params.head()

,state,validation,peak_week,total_cases,peak_height
0,11,1.0,17.912460,13281,864
0,11,2.0,20.624851,5583,604
0,11,3.0,28.967172,2553,142
0,11,4.0,23.444633,1472,72
0,12,1.0,22.992623,4735,288


In [22]:
df_params.to_csv('data/parameters_dengue_state.csv.gz', index=False)